In [39]:
import numpy as np
import pandas as pd
import scipy as sci
import scipy.io as sio
import cfad 
from pathlib import Path

In [41]:
ks_path = Path(cfad.data['Neural']['Renewal']['Concatenated Data']['kilosort4']['Mouse 2'])
output_path = Path(r"Z:\Saij\ephys alignment\Mouse 2")
triggers_path = Path(cfad.data['Neural']['Renewal']['Triggers']['Checkerboard']["Mouse 2"])

In [44]:
triggers = np.array(sio.loadmat(triggers_path)["evt"]).flatten()

In [27]:
ks_data = {}
for file_path in ks_path.glob('*.npy'):
    # Get the filename without extension
    # We replace spaces and hyphens with underscores to make them valid Python variable names
    var_name = file_path.stem.replace(' ', '_').replace('-', '_')
    
    try:
        # Load the data
        data = np.array(np.load(file_path))
        # Inject the variable into the global scope
        globals()[var_name] = data
        ks_data[var_name] = data
        print(f"Created variable: {var_name}")
        
    except Exception as e:
        print(f"Failed to load {file_path.name}: {e}")

# You can now access the variables directly by name
# Example: If you had 'experiment_1.npy', you can now type:
# print(experiment_1.shape)

Created variable: amplitudes
Created variable: channel_map
Created variable: whitening_mat_inv
Created variable: kept_spikes
Created variable: spike_times
Created variable: pc_features
Created variable: templates_ind
Created variable: spike_positions
Created variable: similar_templates
Created variable: spike_detection_templates
Created variable: whitening_mat_dat
Created variable: spike_templates
Created variable: whitening_mat
Created variable: spike_clusters
Created variable: channel_positions
Created variable: channel_shanks
Created variable: pc_feature_ind
Created variable: templates
Failed to load ops.npy: Object arrays cannot be loaded when allow_pickle=False


In [ ]:
print(templates)

In [54]:
def kstoibl(data, outputdir, triggers):
    fs = 30000
    buffer = 2 * 30000
    start = triggers[1] - buffer
    end = triggers[-1] + buffer
    spikes_data_dict = {
        "spikes.times": data["spike_times"],
        "spikes.depths": data["spike_positions"][:, 1],
        "spikes.clusters": data["spike_clusters"],
        "spikes.amps": data["amplitudes"]
        
    }
    df = pd.DataFrame.from_dict(spikes_data_dict)
    print(df.head())
    for i in data:
        if i == "amplitudes":
            np.save(outputdir / "spikes.amps.npy", data[i])
            #print(data[i])
        if i == "spike_clusters":
            np.save(outputdir / "spikes.clusters.npy", data[i])
            #print(data[i])
        if i == "spike_positions":
            depths = data[i][:, 1]
            #print(depths)
            np.save(outputdir / "spikes.depths.npy", data[i])
        if i == "spike_times":
            np.save(outputdir / "spike.times.npy", data[i])
        #if i == "templates":
            #np.save(outputdir / "clusters.peakToTrough")
        
        

In [55]:
kstoibl(ks_data, output_path, triggers)

   spikes.times  spikes.depths  spikes.clusters  spikes.amps
0             4    1479.916138               70    14.550706
1            31    1559.850952               82    14.404827
2            47    3434.364746              256    14.279929
3            80    1691.162964               89    10.897957
4            82    2515.211426              173    13.847266
